# Vessel pipeline — DeepLabV3+ (Run 4)
Standalone Kaggle notebook. Rebuilds the full pipeline from scratch (dataset, loss, metrics, training loop — identical to Runs 1-3) and trains **one new run**: **DeepLabV3+, augmentation ON**, per the professor's requested architecture comparison.

**Before running:**
1. Upload your vessel dataset as a Kaggle Dataset and add it via "+ Add Input".
2. Settings (right panel) → Accelerator → GPU T4x2 or P100 → Internet ON.
3. Update `KAGGLE_INPUT_DIR` in the Locate Dataset cell to match your actual dataset slug.

## Cell 1 — Install dependencies

In [1]:
!apt-get install -y p7zip-full -qq
!pip install -q segmentation-models-pytorch albumentations tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


## Cell 2 — Locate + extract dataset
**Edit `KAGGLE_INPUT_DIR` below** to match your dataset slug, visible in the right-side "Input" panel once added to this notebook.

In [2]:
import os

# TODO: replace with your actual dataset slug
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/zakariaassadiki/vessel-dataset"
LOCAL_ROOT = "/kaggle/working/vessel_data"

os.makedirs(LOCAL_ROOT, exist_ok=True)

print("Contents of Kaggle input dir:")
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    print(root, "->", files[:5], f"({len(files)} files)")

# If you uploaded the raw .7z (not pre-zipped), extract it here:
VESSEL_7Z_PATH = os.path.join(KAGGLE_INPUT_DIR, "vessel_dataset.7z")  # adjust filename
if os.path.exists(VESSEL_7Z_PATH):
    !7z x "{VESSEL_7Z_PATH}" -o{LOCAL_ROOT} -y
else:
    # already extracted by Kaggle (uploaded as .zip) -- just point LOCAL_ROOT at the input dir directly
    LOCAL_ROOT = KAGGLE_INPUT_DIR
    print(f"No .7z found -- assuming Kaggle already extracted your zip. Using LOCAL_ROOT = {LOCAL_ROOT}")

Contents of Kaggle input dir:
/kaggle/input/datasets/zakariaassadiki/vessel-dataset -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/masks -> ['mask_1713.tiff', 'mask_3905.tiff', 'mask_990.tiff', 'mask_2638.tiff', 'mask_5545.tiff'] (6544 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/patches -> ['patch_469.tiff', 'patch_2882.tiff', 'patch_4574.tiff', 'patch_2245.tiff', 'patch_3557.tiff'] (6544 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_masks -> ['mask_291.tiff', 'mask_37.tiff', 'mask_216.tiff', 'mask_183.tiff', 'mask_539.tiff'] (728 files)
/kaggle/input/

## Cell 3 — Inspect extracted structure

In [3]:
for root, dirs, files in os.walk(LOCAL_ROOT):
    depth = root.replace(LOCAL_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/  ({len(files)} files)")
    if files:
        print(f"{indent}  sample: {files[:5]}")
    if depth > 2:
        continue

from collections import Counter
ext_counter = Counter()
for root, dirs, files in os.walk(LOCAL_ROOT):
    for f in files:
        ext_counter[os.path.splitext(f)[1].lower()] += 1
print("\nExtension counts:", ext_counter)

vessel-dataset/  (0 files)
  VascularModelDataset/  (0 files)
    Training/  (0 files)
      masks/  (6544 files)
        sample: ['mask_1713.tiff', 'mask_3905.tiff', 'mask_990.tiff', 'mask_2638.tiff', 'mask_5545.tiff']
      patches/  (6544 files)
        sample: ['patch_469.tiff', 'patch_2882.tiff', 'patch_4574.tiff', 'patch_2245.tiff', 'patch_3557.tiff']
    Testing/  (0 files)
      testset_masks/  (728 files)
        sample: ['mask_291.tiff', 'mask_37.tiff', 'mask_216.tiff', 'mask_183.tiff', 'mask_539.tiff']
      testset_images/  (728 files)
        sample: ['patch_469.tiff', 'patch_242.tiff', 'patch_620.tiff', 'patch_710.tiff', 'patch_228.tiff']
    WSI Annotations/  (4 files)
      sample: ['SSES2021 10 LIVER 12 7 21_J-21-153_4_Pig_GS 1-10000 + PSR_RUN12__Liver LL 0h_VAB.ndpi.ndpa', 'SSES2021 14 LIVER LL 14 7 21_J-21-156_4_Pig_GS 1-10000 + PSR_RUN12__Liver LL 0h_VAB.ndpi.ndpa', 'SSES2021 12 LIVER LL 13 7 21_J-21-155_6_Pig_GS 1-10000 + PSR_RUN12__Liver LL 0h_VAB.ndpi.ndpa', 'SSE

## Cell 4 — Build (image, mask) file pairs
Expected structure: `VascularModelDataset/Training/{patches,masks}/`, `Testing/{testset_images,testset_masks}/`, paired by numeric ID.

In [4]:
DATASET_ROOT = os.path.join(LOCAL_ROOT, "VascularModelDataset")
TRAINING_PATCHES_DIR = os.path.join(DATASET_ROOT, "Training", "patches")
TRAINING_MASKS_DIR = os.path.join(DATASET_ROOT, "Training", "masks")
TEST_IMAGES_DIR = os.path.join(DATASET_ROOT, "Testing", "testset_images")
TEST_MASKS_DIR = os.path.join(DATASET_ROOT, "Testing", "testset_masks")

import glob
import re

def pair_by_id(images_dir, masks_dir, img_prefix="patch", mask_prefix="mask"):
    pairs = []
    missing = []
    for img_path in sorted(glob.glob(os.path.join(images_dir, f"{img_prefix}_*.tiff"))):
        fname = os.path.basename(img_path)
        m = re.match(rf"{img_prefix}_(\d+)\.tiff", fname)
        if not m:
            continue
        num = m.group(1)
        mask_path = os.path.join(masks_dir, f"{mask_prefix}_{num}.tiff")
        if os.path.exists(mask_path):
            pairs.append((img_path, mask_path))
        else:
            missing.append(fname)
    if missing:
        print(f"WARNING: {len(missing)} images had no matching mask, e.g. {missing[:5]}")
    return pairs

training_pairs = pair_by_id(TRAINING_PATCHES_DIR, TRAINING_MASKS_DIR)
test_pairs = pair_by_id(TEST_IMAGES_DIR, TEST_MASKS_DIR)

print(f"Training pairs found: {len(training_pairs)}  (expect ~6544)")
print(f"Test pairs found: {len(test_pairs)}  (expect ~728)")
print("Example training pair:", training_pairs[0])
print("Example test pair:", test_pairs[0])

Training pairs found: 6544  (expect ~6544)
Test pairs found: 728  (expect ~728)
Example training pair: ('/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/patches/patch_1.tiff', '/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/masks/mask_1.tiff')
Example test pair: ('/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_images/patch_1.tiff', '/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_masks/mask_1.tiff')


## Cell 5 — Train/val split (random, fixed seed, per project guide Section 3)

In [5]:
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(training_pairs, test_size=0.2, random_state=42)
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}")

Train: 5235  Val: 1309


## Cell 6 — Dataset class (vessel, 5-class, online augmentation toggle)

In [6]:
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 0 background, 1 portal vein, 2 central vein, 3 hepatic artery, 4 bile duct
NUM_CLASSES = 5
PATCH_SIZE = 256

def get_train_transforms(aug_on: bool):
    if aug_on:
        return A.Compose([
            A.Rotate(limit=180, border_mode=cv2.BORDER_REFLECT, p=0.7),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ElasticTransform(alpha=1, sigma=20, p=0.3),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.4),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

def get_eval_transforms():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class VesselDataset(Dataset):
    # Masks are RGB with 5 core colors + anti-aliased blends at class boundaries.
    # Nearest-color classification snaps each blended edge pixel to its closest
    # pure class color.
    REFERENCE_COLORS = np.array([
        [0, 0, 0],       # 0 background (black)
        [0, 0, 255],     # 1 portal vein (blue)
        [0, 255, 0],     # 2 central vein (green)
        [255, 0, 0],     # 3 artery (red)
        [128, 0, 128],   # 4 bile duct (purple)
    ], dtype=np.float32)

    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    @staticmethod
    def rgb_mask_to_class_index(mask_rgb):
        h, w, _ = mask_rgb.shape
        flat = mask_rgb.reshape(-1, 3).astype(np.float32)
        dists = np.linalg.norm(
            flat[:, None, :] - VesselDataset.REFERENCE_COLORS[None, :, :], axis=2
        )
        class_idx = np.argmin(dists, axis=1)
        return class_idx.reshape(h, w).astype(np.uint8)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)

        mask_bgr = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        mask_rgb = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2RGB)

        # Resize first (nearest-neighbor), THEN color-match on the small array --
        # 16x cheaper than matching at full 1024x1024 resolution.
        mask_rgb_small = cv2.resize(mask_rgb, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = self.rgb_mask_to_class_index(mask_rgb_small)

        transformed = self.transform(image=image, mask=mask)
        image_t = transformed["image"]
        mask_t = transformed["mask"].long()

        return image_t, mask_t

## Cell 7 — Loss: Dice + Focal

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_onehot, dims)
        cardinality = torch.sum(probs + targets_onehot, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

class DiceFocalLoss(nn.Module):
    def __init__(self, num_classes, dice_weight=0.5, focal_weight=0.5):
        super().__init__()
        self.dice = DiceLoss(num_classes)
        self.focal = FocalLoss()
        self.dw = dice_weight
        self.fw = focal_weight

    def forward(self, logits, targets):
        return self.dw * self.dice(logits, targets) + self.fw * self.focal(logits, targets)

## Cell 8 — Per-class Dice metric

In [8]:
def per_class_dice(logits, targets, num_classes, smooth=1e-6):
    preds = torch.argmax(logits, dim=1)
    dices = []
    for c in range(num_classes):
        pred_c = (preds == c).float()
        target_c = (targets == c).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dice_c = (2. * intersection + smooth) / (union + smooth)
        dices.append(dice_c.item())
    return dices

## Cell 9 — Attention U-Net (kept for compatibility with the model factory / future runs)
Not used for this DeepLabV3+ run, but `build_model()` below references it, so it's included for a self-contained notebook.

In [9]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class AttentionGate(nn.Module):
    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(gate_ch, inter_ch, 1),
            nn.BatchNorm2d(inter_ch),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(skip_ch, inter_ch, 1),
            nn.BatchNorm2d(inter_ch),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, 1),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode="bilinear", align_corners=False)
        attn = self.relu(g1 + x1)
        attn = self.psi(attn)
        return x * attn

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=NUM_CLASSES, base_ch=32):
        super().__init__()
        chs = [base_ch, base_ch * 2, base_ch * 4, base_ch * 8, base_ch * 16]

        self.enc1 = ConvBlock(in_channels, chs[0])
        self.enc2 = ConvBlock(chs[0], chs[1])
        self.enc3 = ConvBlock(chs[1], chs[2])
        self.enc4 = ConvBlock(chs[2], chs[3])
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(chs[3], chs[4])

        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2)
        self.att4 = AttentionGate(chs[3], chs[3], chs[3] // 2)
        self.dec4 = ConvBlock(chs[4], chs[3])

        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2)
        self.att3 = AttentionGate(chs[2], chs[2], chs[2] // 2)
        self.dec3 = ConvBlock(chs[3], chs[2])

        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2)
        self.att2 = AttentionGate(chs[1], chs[1], chs[1] // 2)
        self.dec2 = ConvBlock(chs[2], chs[1])

        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2)
        self.att1 = AttentionGate(chs[0], chs[0], chs[0] // 2)
        self.dec1 = ConvBlock(chs[1], chs[0])

        self.out_conv = nn.Conv2d(chs[0], num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        e4_att = self.att4(d4, e4)
        d4 = self.dec4(torch.cat([d4, e4_att], dim=1))

        d3 = self.up3(d4)
        e3_att = self.att3(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3_att], dim=1))

        d2 = self.up2(d3)
        e2_att = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2_att], dim=1))

        d1 = self.up1(d2)
        e1_att = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1_att], dim=1))

        return self.out_conv(d1)

## Cell 10 — Model factory
Includes `deeplabv3plus` (this run) alongside `unetplusplus` / `attention_unet` (kept for compatibility / future runs).

In [10]:
import segmentation_models_pytorch as smp

CLASS_NAMES = ["background", "portal_vein", "central_vein", "artery", "bile_duct"]

def build_model(architecture: str, num_classes: int = NUM_CLASSES):
    if architecture == "unetplusplus":
        return smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights="imagenet",
                                 in_channels=3, classes=num_classes)
    elif architecture == "attention_unet":
        return AttentionUNet(in_channels=3, num_classes=num_classes)
    elif architecture == "deeplabv3plus":
        # Same ResNet34 / ImageNet-pretrained encoder as the U-Net++ baseline --
        # keeps this an apples-to-apples architecture comparison (only the
        # decoder differs: ASPP + atrous convolutions instead of nested skips).
        return smp.DeepLabV3Plus(encoder_name="resnet34", encoder_weights="imagenet",
                                  in_channels=3, classes=num_classes)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

## Cell 11 — Training loop

In [11]:
import time
import csv
import pandas as pd
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

def train_run(run_name, architecture, aug_on, epochs, batch_size, lr=5e-4, max_batches=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"=== {run_name} | arch={architecture} aug_on={aug_on} epochs={epochs} bs={batch_size} | device={device} ===")

    train_ds = VesselDataset(train_pairs, get_train_transforms(aug_on))
    val_ds = VesselDataset(val_pairs, get_eval_transforms())

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True,
                               persistent_workers=True, prefetch_factor=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True,
                             persistent_workers=True, prefetch_factor=2)

    model = build_model(architecture).to(device)
    criterion = DiceFocalLoss(NUM_CLASSES)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()

    checkpoint_dir = "/kaggle/working/checkpoints"
    log_dir = "/kaggle/working/logs"
    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    log_path = f"{log_dir}/{run_name}.csv"

    log_rows = []
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc=f"[{run_name}] epoch {epoch+1}/{epochs} train", leave=False)
        for batch_idx, (images, masks) in enumerate(train_bar):
            if max_batches is not None and batch_idx >= max_batches:
                break
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")
        n_train_batches = min(len(train_loader), max_batches) if max_batches else len(train_loader)
        train_loss /= n_train_batches

        model.eval()
        val_loss = 0.0
        class_dice_sums = [0.0] * NUM_CLASSES
        n_batches = 0
        val_bar = tqdm(val_loader, desc=f"[{run_name}] epoch {epoch+1}/{epochs} val", leave=False)
        with torch.no_grad():
            for batch_idx, (images, masks) in enumerate(val_bar):
                if max_batches is not None and batch_idx >= max_batches:
                    break
                images, masks = images.to(device), masks.to(device)
                with autocast():
                    logits = model(images)
                    loss = criterion(logits, masks)
                val_loss += loss.item()
                dices = per_class_dice(logits, masks, NUM_CLASSES)
                for c in range(NUM_CLASSES):
                    class_dice_sums[c] += dices[c]
                n_batches += 1
        val_loss /= n_batches
        class_dice_avg = [s / n_batches for s in class_dice_sums]

        scheduler.step()

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            row[f"dice_{name}"] = class_dice_avg[c]
        log_rows.append(row)

        print(f"epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | "
              f"dice(bg/pv/cv/art/bd) = {[round(d,3) for d in class_dice_avg]}")

        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            ckpt_path = f"{checkpoint_dir}/{run_name}.pth"
            torch.save(model.state_dict(), ckpt_path)

        pd.DataFrame(log_rows).to_csv(log_path, index=False)

    total_time = time.time() - start_time
    print(f"=== {run_name} done in {total_time/60:.1f} min ===")

    return {
        "run_name": run_name,
        "architecture": architecture,
        "aug_on": aug_on,
        "epochs": epochs,
        "batch_size": batch_size,
        "final_val_loss": val_loss,
        "final_dice_overall": sum(class_dice_avg[1:]) / (NUM_CLASSES - 1),
        **{f"final_dice_{name}": class_dice_avg[c] for c, name in enumerate(CLASS_NAMES)},
        "train_time_min": total_time / 60,
    }

## Run 4 — DeepLabV3+ (teacher-requested comparison), augmentation ON
Same loss (Dice+Focal), same 18 epochs / batch 16 / 256×256 setup as Runs 1-3, so it's directly comparable in the final table. Pretrained encoder like the U-Net++ baseline (Runs 1-2) — unlike Run 3's Attention U-Net, so there's no pretrained-vs-scratch confound here.

**Note:** DeepLabV3+'s ASPP module tends to use more GPU memory than U-Net++ — drop `batch_size` to 8 if this OOMs on your Kaggle T4/P100.

In [12]:
result_run4 = train_run(
    run_name="vessel_run4_deeplabv3plus_augON",
    architecture="deeplabv3plus",
    aug_on=True,
    epochs=18,
    batch_size=16,  # drop to 8 if this OOMs
)

=== vessel_run4_deeplabv3plus_augON | arch=deeplabv3plus aug_on=True epochs=18 bs=16 | device=cuda ===


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

/tmp/ipykernel_58/1128972946.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[vessel_run4_deeplabv3plus_augON] epoch 1/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

...

[vessel_run4_deeplabv3plus_augON] epoch 17/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 17/18 | train_loss 0.1739 | val_loss 0.1686 | dice(bg/pv/cv/art/bd) = [0.981, 0.777, 0.783, 0.704, 0.62]


[vessel_run4_deeplabv3plus_augON] epoch 18/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

[vessel_run4_deeplabv3plus_augON] epoch 18/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 18/18 | train_loss 0.1715 | val_loss 0.1697 | dice(bg/pv/cv/art/bd) = [0.98, 0.773, 0.781, 0.701, 0.621]
=== vessel_run4_deeplabv3plus_augON done in 59.0 min ===


## Save run log
This notebook only trains Run 4. To merge with your existing Runs 1-3 log, combine this CSV with your other notebook's `runs_log_vessel.csv` afterward (e.g. `pd.concat([...])`) before building the final comparison table.

In [13]:
runs_log_run4 = pd.DataFrame([result_run4])
runs_log_path = "/kaggle/working/runs_log_vessel_run4.csv"
runs_log_run4.to_csv(runs_log_path, index=False)
print(f"Saved to {runs_log_path}")
runs_log_run4

Saved to /kaggle/working/runs_log_vessel_run4.csv


,run_name,architecture,aug_on,epochs,batch_size,final_val_loss,final_dice_overall,final_dice_background,final_dice_portal_vein,final_dice_central_vein,final_dice_artery,final_dice_bile_duct,train_time_min
0,vessel_run4_deeplabv3plus_augON,deeplabv3plus,True,18,16,0.169708,0.718901,0.979903,0.772579,0.781348,0.701014,0.620664,59.005123


## Zip + download
Download from the Kaggle Output panel, then merge into your shared Drive folder alongside the Runs 1-3 results, per the project guide's Kaggle↔Drive workflow.

In [14]:
!zip -r /kaggle/working/vessel_run4_results.zip /kaggle/working/checkpoints /kaggle/working/logs /kaggle/working/runs_log_vessel_run4.csv

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/vessel_run4_deeplabv3plus_augON.pth (deflated 7%)
  adding: kaggle/working/logs/ (stored 0%)
  adding: kaggle/working/logs/vessel_run4_deeplabv3plus_augON.csv (deflated 50%)
  adding: kaggle/working/runs_log_vessel_run4.csv (deflated 39%)
